In [1]:
# Celda 1
%cd C:/Users/fabi_/Documents/workflow/textos-catolicos-xix

import os
import re
from collections import Counter
import pandas as pd
from spylls.hunspell import Dictionary  # Spanish Hunspell dictionary in pure Python

BASE_FILE = "metadatos/textos-base.csv"     # Hand-curated: one row per text
AUTHORS_FILE = "metadatos/autores.csv"      # Hand-curated: one row per author
CLEAN_FOLDER = "textos-limpios"             # From ocr-piloto/5-limpieza
TEXTS_FILE = "metadatos/textos.csv"         # Output: the work and its author
EDITIONS_FILE = "metadatos/ediciones.csv"   # Output: the edition each text comes from
EXCEL_FILE = "metadatos/catalogo.xlsx"      # Output: the same tables, for reading and filtering by hand
DICTIONARY = "ocr-piloto/diccionario-es/es_ES"
MIN_BOOKS = 10  # A word in 10 books or more of the corpus exists, even if the modern dictionary lacks it ("dixo")

C:\Users\fabi_\Documents\workflow\textos-catolicos-xix


In [2]:
# Celda 2
# The columns of the first author go into the table of texts, so that a filter needs no merge:
# texts[texts["author_institution"] == "Jesuita"]
base = pd.read_csv(BASE_FILE, dtype=str, keep_default_na=False)
authors = pd.read_csv(AUTHORS_FILE, dtype=str, keep_default_na=False).set_index("author_id")

first_author = base["authors"].str.split(";").str[0]
texts = base[["text_id", "work_id", "title", "volume"]].copy()
# "nieremberg;andrade;casani" -> "Juan Eusebio Nieremberg; Alonso de Andrade; José Casani":
texts["author"] = base["authors"].map(lambda ids: "; ".join(authors.loc[i, "author"] for i in ids.split(";")))
texts["author_institution"] = first_author.map(authors["institution"])
texts["author_gender"] = first_author.map(authors["gender"])
texts["author_death_period"] = first_author.map(authors["death_period"])
texts["author_in_the_americas"] = first_author.map(authors["the_americas"])
for column in ["authorship", "topic", "genre", "original_language", "translator", "first_edition", "first_edition_source"]:
    texts[column] = base[column]
texts["first_edition"] = pd.to_numeric(texts["first_edition"]).astype("Int64")
texts[["author", "author_institution"]].value_counts().head(8)

author                                                    author_institution
Luis de Granada                                           Dominico              11
Teresa de Ávila                                           Carmelita             11
María de Jesús de Ágreda                                  Concepcionista         9
Juan de Ávila                                             Clero secular          9
Juan Eusebio Nieremberg; Alonso de Andrade; José Cassani  Jesuita                9
Juan de la Cruz                                           Carmelita              8
Juan Eusebio Nieremberg                                   Jesuita                8
Juan de los Ángeles                                       Franciscano            7
Name: count, dtype: int64

In [3]:
# Celda 3
# Size and OCR quality of each text. A word "does not exist" if it is neither in the Spanish dictionary nor in 10 books
# or more of the corpus (as in 7-paginas-baja-calidad). A few minutes:
WORD = re.compile(r"[^\W\d_]+")

def book_words(path):
    with open(path, encoding="utf-8") as f:
        text = re.sub(r"<[^>]+>", " ", f.read())
    return [word.lower() for word in WORD.findall(text)]

words_by_book = {text_id: book_words(os.path.join(CLEAN_FOLDER, text_id + ".txt")) for text_id in texts["text_id"]}
books_per_word = Counter()
for words in words_by_book.values():
    books_per_word.update(set(words))
dictionary = Dictionary.from_files(DICTIONARY)
exists = {word: books_per_word[word] >= MIN_BOOKS or dictionary.lookup(word) for word in books_per_word}

texts["words"] = [len(words_by_book[text_id]) for text_id in texts["text_id"]]
unknown_pct = [round(100 * sum(1 for word in words_by_book[text_id] if not exists[word]) / max(len(words_by_book[text_id]), 1), 2)
               for text_id in texts["text_id"]]
texts["words"].describe().round()

count       151.0
mean     113489.0
std       98657.0
min        4847.0
25%       35855.0
50%       80338.0
75%      167406.0
max      626039.0
Name: words, dtype: float64

In [4]:
# Celda 4
# The two tables. texts: the work; editions: where its text comes from.
texts["clean_file"] = "textos-limpios/" + texts["text_id"] + ".txt"
texts["modern_file"] = "textos-modernizados/" + texts["text_id"] + ".txt"
texts.to_csv(TEXTS_FILE, index=False, encoding="utf-8", lineterminator="\n")

editions = base[["text_id", "edition_year", "edition_city", "edition_country", "printer", "collection", "editor", "modern_edition",
                 "repository", "link", "pdf_pages", "pages_text"]].copy()
editions["unknown_words_pct"] = unknown_pct
editions.to_csv(EDITIONS_FILE, index=False, encoding="utf-8", lineterminator="\n")
print(texts.shape, editions.shape)

(151, 19) (151, 13)


In [5]:
# Celda 5
# What each column means, in the Excel file:
COLUMNS = [
    ("text_id", "File name without extension: author, title and volume", "Nombre del archivo sin extensión: autor, título y tomo"),
    ("work_id", "The same work in all its volumes", "La misma obra en todos sus tomos"),
    ("title / volume", "Short title of the work; volume, if the work is split", "Título breve de la obra; tomo, si la obra está dividida"),
    ("author", "Author or authors, separated by ;", "Autor o autores, separados por ;"),
    ("author_*", "Of the first author: religious order or state, gender, period of death, whether he or she lived in the Americas",
     "Del primer autor: orden religiosa o estado, género, período de muerte, si vivió en América"),
    ("authorship", "segura, atribuida or varios autores", "segura, atribuida o varios autores"),
    ("topic", "Broad subject, with the vocabulary of the Jesuit libraries dashboard", "Tema general, con el vocabulario del dashboard de bibliotecas jesuíticas"),
    ("genre", "Form of the text (Tratado, Vida, Cartas...)", "Forma del texto (Tratado, Vida, Cartas...)"),
    ("original_language / translator", "Language in which the work was written; translator into Spanish", "Lengua en que se escribió la obra; traductor al español"),
    ("first_edition / first_edition_source", "Year of the first printed edition, in the original language; catalog record (a note in parentheses if it covers only part of the text)",
     "Año de la primera edición impresa, en la lengua original; registro de catálogo (con una nota entre paréntesis si cubre solo parte del texto)"),
    ("words", "Words of the text", "Palabras del texto"),
    ("edition_* / printer / collection / editor", "The 19th- or 20th-century edition used", "La edición de los siglos XIX o XX usada"),
    ("modern_edition", "Filled in if the source is a modern edition", "Con dato si la fuente es una edición moderna"),
    ("repository / link / pdf_pages / pages_text", "Where the PDF comes from; pages of the PDF and of the text", "De dónde viene el PDF; páginas del PDF y del texto"),
    ("unknown_words_pct", "Share of words found neither in the dictionary nor in 10 books: a measure of OCR quality",
     "Porcentaje de palabras que no están en el diccionario ni en 10 libros: una medida de la calidad del OCR"),
]
columns_sheet = pd.DataFrame(COLUMNS, columns=["column", "English", "Español"])
authors_sheet = pd.read_csv(AUTHORS_FILE, dtype=str, keep_default_na=False)
with pd.ExcelWriter(EXCEL_FILE, engine="openpyxl") as writer:
    for name, table in [("textos", texts), ("ediciones", editions), ("autores", authors_sheet), ("columnas", columns_sheet)]:
        table.to_excel(writer, sheet_name=name, index=False)
        sheet = writer.sheets[name]
        sheet.freeze_panes = "B2"
        sheet.auto_filter.ref = sheet.dimensions
print("saved", EXCEL_FILE)

saved metadatos/catalogo.xlsx
